In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os

def clean():
    folder_paths = ["logs"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": "local",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 23:27:37,344 [DEBUG] [Rain] Rain is initialized
2023-07-03 23:27:37,351 [DEBUG] [Provisioner] Creating coordinator
2023-07-03 23:27:37,356 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\coord/
2023-07-03 23:27:37,362 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 23:27:37,368 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 23:27:37,373 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\coord/
2023-07-03 23:27:37,377 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized
2023-07-03 23:27:37,383 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\divider/
2023-07-03 23:27:37,391 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\divider/
2023-07-03 23:27:37,397 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\divider_proxy/
2023-07-03 23:27:37,406 [DEBUG] [Temporar

In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 23:27:37,610 [DEBUG] [Rain] Creating workers
2023-07-03 23:27:37,665 [INFO] [Provisioner] provisioner is serving
2023-07-03 23:27:37,671 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 23:27:37,695 [INFO] [Coordinator] coordinator is serving
2023-07-03 23:27:37,698 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 23:27:37,734 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 23:27:37,749 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 23:27:37,754 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 23:27:37,763 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\worker/
2023-07-03 23:27:37,788 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 23:27:37,792 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\worker/
2023-07-03 23:27:37,803 [INFO] [W

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

AttributeError: 'NoneType' object has no attribute 'evaluate'

In [ ]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 23:08:34,260 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-03 23:08:34,277 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-03 23:08:34,282 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-03 23:08:34,289 [ERROR] [Coordinator] Error in the coordinator server: Failed to bind to address [::]:50052; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
ERROR:Coordinator:Error in the coordinator server: Failed to bind to address [::]:50052; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
2023-07-03 23:08:34,296 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 workers
2023-07-03 23:08:34,305 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data\worker/
DEBUG:TemporaryFilesManager:Creating temporary directory ../../../../Rain/data\worker/
2023-07-03 23:08:34,313 [ERR

KeyboardInterrupt: 

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0985 - accuracy: 0.9711

Test accuracy: 97.1%
